# Step 1 · Fit the models and check them on held-out data

> **Status: written for review. This notebook has not been run yet.**

**Project:** comparing how explainable black-box models are when they detect anonymous fake accounts (bots and trolls) on social media. This notebook is **step 1**: fit every model on the training data, test it on the held-out test data, and store the model, its predictions and its accuracy measures. **Step 2** (not here) runs explainability methods on the stored models.

Because explainability matters more than accuracy, the goal here is *a model that learned something real*, not a leaderboard. Each model only needs to be good enough that explaining it is meaningful.

## Conventions used by every model

| Convention | What it means |
|---|---|
| **Task** | Binary: `label = 1` for an inauthentic account (bot or troll), `0` for an authentic one. |
| **Splits** | The `split` column of `accounts.csv` (60 / 20 / 20, stratified). Models are **fit on `train`**. `val` is used for early stopping and for choosing the decision threshold. `test` is only used for the final numbers. |
| **Score** | Every account-level model outputs one number per account where **higher = more likely inauthentic**. `score_type` says what it is: `probability` (a classifier probability), `anomaly` (an unsupervised anomaly score) or `score` (an uncalibrated graph-propagation belief). Only `probability` scores get log loss and Brier score. |
| **Threshold** | One rule for all models: the score cut-off that maximises F1 on `val`. It is never tuned on `test`. |
| **Unit** | Every model predicts per **account**. |
| **Truth files** | Nothing in `truth/` is given to any model. The only use is a reference line (the best possible AUC) in the final summary. |
| **Failures** | Each model runs inside a wrapper. If one fails (for example, a missing library), the error is recorded and the notebook continues. |

## Models in this notebook (28)

Rows 11 (Siamese network), 27 (pretrained toxicity classifier) and 29 (adversarial robustness) of the methods sheet are intentionally excluded.

| Sheet row | Model | Library | Input |
|---|---|---|---|
| 6 | Random forest | scikit-learn | 12 profile stats |
| 7 | XGBoost, LightGBM | `xgboost`, `lightgbm` | 12 profile stats |
| 8 | SVM (RBF kernel), SVM (linear) | scikit-learn | 12 profile stats |
| 9 | 1D-CNN, RNN, LSTM | PyTorch | 168-hour posting timeline |
| 10 | Small transformer | PyTorch | Post text |
| 12 | GCN, GAT, GraphSAGE, R-GCN | PyTorch Geometric | Graph + profile stats |
| 13 | Dense autoencoder, LSTM autoencoder | PyTorch | Profile stats; posting timeline |
| 14 | Multimodal fusion | PyTorch + PyTorch Geometric | Text embedding + profile stats + graph |
| 19 | PCA, NMF, truncated SVD (each + logistic regression) | scikit-learn | 12 profile stats |
| 20 | Isolation Forest, LOF, one-class SVM | scikit-learn | 12 profile stats |
| 21 | word2vec, doc2vec, node2vec, DeepWalk (each + logistic regression) | `gensim`, `node2vec` | Post text; graph |
| 22 | Label spreading, belief propagation (SybilBelief-style) | NumPy / SciPy | Graph + a few known labels |

Where a model departs from the published method it is named after (different input, simplified architecture, no pretrained weights), the markdown above that model says so.

## 0 · Setup

In [ ]:
# Install what the notebook needs (uncomment once). PyTorch Geometric has its own install page:
# https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
#
# %pip install -r ../requirements.txt
# %pip install xgboost lightgbm gensim node2vec networkx joblib matplotlib scipy
# %pip install torch torch_geometric

In [ ]:
import json
import platform
import subprocess
import time
import traceback
import warnings
from importlib import metadata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy import sparse
from sklearn.decomposition import NMF, PCA, TruncatedSVD
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC, OneClassSVM

# Libraries that only some models need (xgboost, lightgbm, gensim, node2vec, torch_geometric)
# are imported inside those models' own functions, so a missing one only fails that model.

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# ---------------------------------------------------------------- configuration
SEED = 0  # one seed for everything; each model re-seeds before it is fitted
QUICK = False  # True = tiny epochs / fewer trees, to smoke-test the notebook in a few minutes

# Paths. The notebook lives in notebooks/, so find the repo root from there or from the repo itself.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "synthetic"  # python -m anonsynth --out data/synthetic
OUT_DIR = REPO_ROOT / "outputs" / "step1_fit"
MODEL_DIR = OUT_DIR / "models"
PRED_DIR = OUT_DIR / "predictions"
for d in (MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Training budgets
N_EST = 50 if QUICK else 500  # trees in forests / boosters
EPOCHS = 3 if QUICK else 40  # neural networks trained in mini-batches
GNN_EPOCHS = 20 if QUICK else 200  # full-batch graph models
EMB_EPOCHS = 3 if QUICK else 30  # word2vec / doc2vec / node2vec
N2V_WALKS = 3 if QUICK else 10  # random walks per node

# Label propagation starts from a small set of known labels (as in the sybil-detection literature)
LP_LABELED_FRAC = 0.2  # share of TRAIN accounts whose label is revealed to the propagation models

# Status flags in the summary table (test ROC-AUC)
CHANCE_BAND, WEAK_BAND = 0.55, 0.65

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print("device:", DEVICE, "| quick mode:", QUICK)

In [ ]:
# ---------------------------------------------------------------- provenance
# Recorded with every run so results can be traced to the data version and library versions.
def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def git(*args):
    try:
        out = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, timeout=10)
        return out.stdout.strip() or None
    except Exception:
        return None


ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": git("rev-parse", "HEAD"),
    "git_describe": git("describe", "--tags", "--always"),
    "seed": SEED,
    "quick": QUICK,
    "packages": {
        p: pkg_version(p)
        for p in ["numpy", "pandas", "scikit-learn", "scipy", "torch", "torch-geometric",
                  "xgboost", "lightgbm", "gensim", "node2vec", "networkx"]
    },
}
print(json.dumps(ENV, indent=2))

## 1 · Load the data and build the input views

Each model family sees a different *view* of the same accounts. All views are built from the model-facing files only (`accounts`, `posts`, `edges`, `sequences`), and every scaler or vocabulary is **fit on the training accounts only**.

| View | Built from | Used by |
|---|---|---|
| `X_df` / `X_tab_std` | 12 profile stats (raw / standardised) | forests, boosting, SVMs, autoencoder, decomposition, anomaly detectors, GNN node features |
| `X_seq` | 168 hourly post counts, `log1p` then standardised | CNN, RNN, LSTM, LSTM autoencoder |
| tokens | the 6 posts of each account, split on whitespace | transformer, word2vec, doc2vec |
| `src`, `dst` | undirected edge list | GNNs, node2vec, DeepWalk, label propagation |

Raw (unscaled) features are kept for the tree models and as the explanation space for step 2, so later feature-importance results are in the original units.

In [ ]:
required = ["accounts.csv", "posts.csv", "edges.csv", "sequences.npz"]
missing = [f for f in required if not (DATA_DIR / f).exists()]
assert not missing, f"Missing {missing} in {DATA_DIR}. Generate them with: python -m anonsynth --out data/synthetic"

accounts = pd.read_csv(DATA_DIR / "accounts.csv")
posts = pd.read_csv(DATA_DIR / "posts.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")
_seq = np.load(DATA_DIR / "sequences.npz")

# --- alignment checks: every per-account table must be in account_id order 0..n-1
n = len(accounts)
assert (accounts["account_id"].to_numpy() == np.arange(n)).all()
assert (_seq["account_id"] == np.arange(n)).all()

# --- labels and splits
y = accounts["label"].to_numpy().astype(int)
split = accounts["split"].to_numpy()
is_tr, is_va, is_te = (split == "train"), (split == "val"), (split == "test")
print("accounts by split:", {s: int(m.sum()) for s, m in [("train", is_tr), ("val", is_va), ("test", is_te)]})
print("share inauthentic :", {s: round(float(y[m].mean()), 3) for s, m in [("train", is_tr), ("val", is_va), ("test", is_te)]})

# --- tabular view
FEATURES = [c for c in accounts.columns if c not in ("account_id", "label", "split")]
X_df = accounts[FEATURES]  # raw features, with names (for trees and for step-2 explanations)
X_tab = X_df.to_numpy(dtype=float)
scaler_tab = StandardScaler().fit(X_tab[is_tr])
X_tab_std = scaler_tab.transform(X_tab)

# --- sequence view: log1p of hourly counts, standardised with train statistics
_seq_log = np.log1p(_seq["counts"].astype(float))
X_seq = (_seq_log - _seq_log[is_tr].mean()) / _seq_log[is_tr].std()

# --- text view
post_tokens = [t.split() for t in posts["text"]]
post_account = posts["account_id"].to_numpy()
account_tokens = [[] for _ in range(n)]  # all tokens of an account's posts, concatenated
for toks, a in zip(post_tokens, post_account):
    account_tokens[a].extend(toks)

# --- graph view (undirected; edges.csv lists each edge once)
src, dst = edges["source"].to_numpy(), edges["target"].to_numpy()
EDGE_INDEX = torch.as_tensor(np.vstack([np.r_[src, dst], np.r_[dst, src]]), dtype=torch.long)

# --- tensors shared by the neural models
T_tab = torch.as_tensor(X_tab_std, dtype=torch.float32)
T_seq_cnn = torch.as_tensor(X_seq, dtype=torch.float32).unsqueeze(1)  # (n, 1, 168): channels first
T_seq_rnn = torch.as_tensor(X_seq, dtype=torch.float32).unsqueeze(-1)  # (n, 168, 1): time first

tr_idx, va_idx, te_idx = (np.flatnonzero(m) for m in (is_tr, is_va, is_te))
tr_t, va_t = torch.as_tensor(tr_idx), torch.as_tensor(va_idx)  # torch copies, for indexing tensors

print(f"{n} accounts | {len(posts)} posts | {len(edges)} edges | {len(FEATURES)} profile features")

In [ ]:
# ---------------------------------------------------------------- reference ceiling (NOT given to any model)
# truth/ holds the answer key. We read exactly one column from it, only to print the best possible
# ROC-AUC for a tabular model (the label is a known function of the profile stats plus randomness).
_ref = pd.read_csv(DATA_DIR / "truth" / "accounts_truth.csv", usecols=["true_prob"])["true_prob"].to_numpy()
CEILING_TEST_AUC = roc_auc_score(y[is_te], _ref[is_te])
del _ref
print(f"Reference: best possible test ROC-AUC from the profile stats alone = {CEILING_TEST_AUC:.3f}")

## 2 · Evaluation and bookkeeping helpers

All 28 models go through the same `register(...)` call, so their outputs have the same format.

**What gets stored for every model**

| Item | Where |
|---|---|
| Fitted model | `outputs/step1_fit/models/<name>.joblib` (scikit-learn, boosting, composites), `.pt` (PyTorch weights; re-create the class from this notebook to reload), `.gensim` (embeddings). Extra objects needed to reuse a model (vocabulary, scalers, downstream classifier) go in `<name>_extra.joblib`. |
| Predictions | `outputs/step1_fit/predictions/<name>.csv`: one row per account for **all splits**, with `y_true`, `score`, `y_pred`, `score_type`. |
| Accuracy measures | `metrics.csv` (see the summary section) with train / val / test values. |
| Provenance | `registry.json`: hyperparameters, input view, training time, notes. `environment.json`: library versions and git commit. |

**Accuracy measures** (computed on the same test accounts for every model)

- **ROC-AUC and PR-AUC**: threshold-free, so they work for probabilities and anomaly scores alike. ROC-AUC is the headline number, with a bootstrap 95% interval.
- **F1, precision, recall, balanced accuracy, accuracy** at the validation-chosen threshold. About 70% of accounts are authentic, so plain accuracy is judged against the "always say authentic" baseline registered below.
- **Log loss and Brier score**: only for models whose score is a probability. They are left empty for anomaly scores.

In [ ]:
RESULTS, PREDS, FAILURES = {}, {}, {}


class Timer:
    """with Timer() as t: ...   then t.seconds"""

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0


def best_f1_threshold(y_true, score):
    """Score cut-off that maximises F1; searched over score quantiles."""
    cands = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 197)))
    f1s = [f1_score(y_true, score >= t, zero_division=0) for t in cands]
    return float(cands[int(np.argmax(f1s))])


def split_metrics(y_true, score, threshold, score_type):
    y_pred = (score >= threshold).astype(int)
    out = {
        "n": int(len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "pr_auc": float(average_precision_score(y_true, score)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "log_loss": np.nan,
        "brier": np.nan,
    }
    if score_type == "probability":  # only meaningful for calibrated-looking probabilities
        p = np.clip(score, 1e-6, 1 - 1e-6)
        out["log_loss"] = float(log_loss(y_true, p, labels=[0, 1]))
        out["brier"] = float(brier_score_loss(y_true, p))
    return out


def bootstrap_auc_ci(y_true, score, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].min() == y_true[idx].max():
            continue
        aucs.append(roc_auc_score(y_true[idx], score[idx]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def save_artifact(obj, stem):
    """Save a fitted model in the format that suits it; returns the file path."""
    stem = Path(stem)
    if isinstance(obj, torch.nn.Module):
        path = stem.with_suffix(".pt")
        torch.save(obj.state_dict(), path)  # weights only; the class is defined in this notebook
    elif type(obj).__module__.startswith("gensim") and hasattr(obj, "save"):
        path = stem.with_suffix(".gensim")
        obj.save(str(path))
    else:
        path = stem.with_suffix(".joblib")
        joblib.dump(obj, path)
    return str(path)


def _register(name, unit, row, family, ids, y_true, split_arr, score, score_type, model,
              input_view, hyperparams, seconds, supervised, notes, extra, threshold):
    score = np.asarray(score, dtype=float)
    assert score.shape == (len(y_true),), f"{name}: expected {len(y_true)} scores, got {score.shape}"
    assert np.isfinite(score).all(), f"{name}: scores contain NaN or inf"

    masks = {s: split_arr == s for s in ("train", "val", "test")}
    thr = best_f1_threshold(y_true[masks["val"]], score[masks["val"]]) if threshold is None else float(threshold)
    metrics = {s: split_metrics(y_true[m], score[m], thr, score_type) for s, m in masks.items()}
    ci_lo, ci_hi = bootstrap_auc_ci(y_true[masks["test"]], score[masks["test"]])

    artifact = save_artifact(model, MODEL_DIR / name) if model is not None else None
    if extra is not None:
        joblib.dump(extra, MODEL_DIR / f"{name}_extra.joblib")

    preds = ids.copy()
    preds.insert(0, "model", name)
    preds["split"] = split_arr
    preds["y_true"] = y_true
    preds["score"] = score
    preds["y_pred"] = (score >= thr).astype(int)
    preds["score_type"] = score_type
    preds.to_csv(PRED_DIR / f"{name}.csv", index=False)
    PREDS[name] = preds

    RESULTS[name] = {
        "row": row, "family": family, "unit": unit, "supervised": supervised, "score_type": score_type,
        "input_view": input_view, "hyperparams": hyperparams, "train_seconds": round(float(seconds), 1),
        "threshold": thr, "metrics": metrics, "test_auc_ci": [ci_lo, ci_hi], "artifact": artifact, "notes": notes,
    }
    t = metrics["test"]
    print(f"  {name:<22} test AUC {t['roc_auc']:.3f} [{ci_lo:.3f}, {ci_hi:.3f}] | "
          f"F1 {t['f1']:.3f} | acc {t['accuracy']:.3f} | val AUC {metrics['val']['roc_auc']:.3f} | {seconds:.0f}s")


def register(name, row, family, score, score_type, model, input_view, hyperparams, seconds,
             supervised=True, notes="", extra=None, threshold=None):
    """Register an ACCOUNT-level model. `score` has one value per account (all splits), higher = more inauthentic."""
    ids = pd.DataFrame({"account_id": accounts["account_id"].to_numpy()})
    _register(name, "account", row, family, ids, y, split, score, score_type, model, input_view,
              hyperparams, seconds, supervised, notes, extra, threshold)


def run(name, fn):
    """Run one model's fitting function; record the error instead of stopping the notebook."""
    print(f"\n=== {name} ===")
    set_seed(SEED)
    try:
        fn()
    except Exception as e:  # noqa: BLE001 - we want every failure recorded, whatever its type
        FAILURES[name] = {"error": repr(e), "traceback": traceback.format_exc()}
        print(f"  [FAILED] {name}: {e!r}")

In [ ]:
# ---------------------------------------------------------------- generic PyTorch training helpers
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


@torch.no_grad()
def predict_logits(model, inputs, batch_size=1024):
    """Run `model(*batch)` over tensors in batches; returns a NumPy array of logits."""
    model.eval().to(DEVICE)
    out = []
    for i in range(0, len(inputs[0]), batch_size):
        out.append(model(*[t[i:i + batch_size].to(DEVICE) for t in inputs]).detach().cpu())
    return torch.cat(out).numpy()


def train_torch(model, train_inputs, y_train, val_inputs, y_val, epochs=EPOCHS, lr=1e-3,
                batch_size=128, patience=8, weight_decay=1e-4, clip=1.0):
    """Mini-batch training with binary cross-entropy and early stopping on validation loss.

    `train_inputs` / `val_inputs` are lists of tensors passed to `model(*inputs)`. The model with the
    best validation loss is restored at the end. Returns a small dict describing the run.
    """
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ytr = torch.as_tensor(y_train, dtype=torch.float32)
    yva = torch.as_tensor(y_val, dtype=torch.float32)
    best, best_state, best_ep, bad = np.inf, None, 0, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(ytr))
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            logits = model(*[t[idx].to(DEVICE) for t in train_inputs])
            loss = F.binary_cross_entropy_with_logits(logits, ytr[idx].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        val_loss = F.binary_cross_entropy_with_logits(
            torch.as_tensor(predict_logits(model, val_inputs)), yva).item()
        if val_loss < best - 1e-4:
            best, best_ep, bad = val_loss, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return {"epochs_run": ep + 1, "best_epoch": best_ep + 1, "best_val_loss": round(float(best), 4)}


def train_fullbatch(model, forward, y_all, tr_mask, va_mask, epochs=GNN_EPOCHS, lr=0.01,
                    weight_decay=5e-4, patience=20):
    """Full-batch training for graph models. `forward(model)` returns one logit per node (all nodes).

    Transductive: the whole graph is visible, but the loss only uses TRAIN node labels. Returns
    (info dict, logits for all nodes) for the model with the best validation loss.
    """
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    yt = torch.as_tensor(y_all, dtype=torch.float32, device=DEVICE)
    trm = torch.as_tensor(tr_mask, device=DEVICE)
    vam = torch.as_tensor(va_mask, device=DEVICE)
    best, best_state, best_ep, bad = np.inf, None, 0, 0
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(forward(model)[trm], yt[trm])
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.binary_cross_entropy_with_logits(forward(model)[vam], yt[vam]).item()
        if val_loss < best - 1e-4:
            best, best_ep, bad = val_loss, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = forward(model).detach().cpu().numpy()
    return {"epochs_run": ep + 1, "best_epoch": best_ep + 1, "best_val_loss": round(float(best), 4)}, logits


def train_autoencoder(model, X_train, X_val, epochs=EPOCHS, lr=1e-3, batch_size=128, patience=8):
    """Unsupervised reconstruction training (no labels). Early stopping on validation reconstruction error."""
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, best_ep, bad = np.inf, None, 0, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X_train))
        for i in range(0, len(perm), batch_size):
            xb = X_train[perm[i:i + batch_size]].to(DEVICE)
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        val_loss = float(recon_error(model, X_val).mean())
        if val_loss < best - 1e-5:
            best, best_ep, bad = val_loss, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return {"epochs_run": ep + 1, "best_epoch": best_ep + 1, "best_val_recon_mse": round(float(best), 5)}


@torch.no_grad()
def recon_error(model, X, batch_size=1024):
    """Per-sample mean squared reconstruction error = the anomaly score."""
    model.eval().to(DEVICE)
    errs = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(DEVICE)
        errs.append(((model(xb) - xb) ** 2).flatten(1).mean(1).cpu())
    return torch.cat(errs).numpy()

### Baseline: "always say authentic"

A model with no information. It gives the accuracy you get for free (about 70%) and an AUC of exactly 0.5, so every real model can be judged against it. Its predicted label is fixed to 0 for every account.

In [ ]:
def fit_baseline():
    with Timer() as tm:
        score = np.zeros(n)  # constant score: carries no information
    register("baseline_all_authentic", row=0, family="Baseline", score=score, score_type="probability",
             model=None, input_view="none", hyperparams={}, seconds=tm.seconds, supervised=False,
             notes="Predicts 'authentic' for everyone; its accuracy is the share of authentic accounts.",
             threshold=0.5)


run("baseline_all_authentic", fit_baseline)

## 3 · Row 6 — Random forest

| | |
|---|---|
| **Literature anchor** | Davis et al. (2016), *BotOrNot* / Botometer, which is a random forest over account features. |
| **Input** | The 12 profile stats, raw (trees do not need scaling, and raw units keep step-2 explanations readable). |
| **Output** | Probability of "inauthentic" (fraction of trees voting for it). |
| **Fidelity** | High for the model family. Our features are synthetic stand-ins for Botometer's account features. |

In [ ]:
def fit_rf():
    hp = dict(n_estimators=N_EST, min_samples_leaf=3, max_features="sqrt")
    with Timer() as tm:
        m = RandomForestClassifier(**hp, n_jobs=-1, random_state=SEED)
        m.fit(X_df.loc[is_tr], y[is_tr])
    register("rf", row=6, family="Supervised classification", score=m.predict_proba(X_df)[:, 1],
             score_type="probability", model=m, input_view="12 profile stats (raw)",
             hyperparams=hp, seconds=tm.seconds)


run("rf", fit_rf)

## 4 · Row 7 — Gradient boosting (XGBoost, LightGBM)

| | |
|---|---|
| **Literature anchor** | Shevtsov et al. (2022), an XGBoost bot detector explained with SHAP. |
| **Input** | The 12 profile stats, raw. |
| **Output** | Probability of "inauthentic". |
| **Training detail** | Up to `N_EST` boosting rounds, with early stopping on the validation set (30 rounds without improvement). |
| **Fidelity** | High for the model family. Both libraries are fitted because the sheet lists both. |

In [ ]:
def fit_xgboost():
    import xgboost as xgb  # imported here so a missing library only fails this model

    hp = dict(n_estimators=N_EST, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              early_stopping_rounds=30)
    with Timer() as tm:
        m = xgb.XGBClassifier(**hp, eval_metric="logloss", n_jobs=-1, random_state=SEED)
        m.fit(X_df.loc[is_tr], y[is_tr], eval_set=[(X_df.loc[is_va], y[is_va])], verbose=False)
    register("xgboost", row=7, family="Supervised classification", score=m.predict_proba(X_df)[:, 1],
             score_type="probability", model=m, input_view="12 profile stats (raw)", hyperparams=hp,
             seconds=tm.seconds, notes=f"best_iteration={getattr(m, 'best_iteration', None)}")


def fit_lightgbm():
    import lightgbm as lgb

    hp = dict(n_estimators=N_EST, num_leaves=15, learning_rate=0.05, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, early_stopping_rounds=30)
    with Timer() as tm:
        m = lgb.LGBMClassifier(n_estimators=hp["n_estimators"], num_leaves=15, learning_rate=0.05,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                               random_state=SEED, n_jobs=-1, verbose=-1)
        m.fit(X_df.loc[is_tr], y[is_tr], eval_set=[(X_df.loc[is_va], y[is_va])],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    register("lightgbm", row=7, family="Supervised classification", score=m.predict_proba(X_df)[:, 1],
             score_type="probability", model=m, input_view="12 profile stats (raw)", hyperparams=hp,
             seconds=tm.seconds, notes=f"best_iteration={m.best_iteration_}")


run("xgboost", fit_xgboost)
run("lightgbm", fit_lightgbm)

## 5 · Row 8 — Support vector machines (RBF kernel and linear)

| | |
|---|---|
| **Literature anchor** | Chun et al. (2019), an SVM troll classifier on IRA tweets (the paper's kernel could not be confirmed, so both kernels are fitted). |
| **Input** | The 12 profile stats, standardised inside the pipeline (SVMs are scale-sensitive). |
| **Output** | Probability of "inauthentic" via Platt scaling (`probability=True`, which cross-validates internally and is slower). |
| **Why two kernels** | The RBF SVM is the black-box case. The linear SVM is a useful contrast: its weights are directly readable, so step 2 can check explanation methods against something transparent. |

In [ ]:
def fit_svm(kernel, name):
    hp = dict(kernel=kernel, C=1.0, gamma="scale")
    with Timer() as tm:
        m = Pipeline([("scale", StandardScaler()),
                      ("svc", SVC(**hp, probability=True, random_state=SEED))])
        m.fit(X_df.loc[is_tr], y[is_tr])
    register(name, row=8, family="Supervised classification", score=m.predict_proba(X_df)[:, 1],
             score_type="probability", model=m, input_view="12 profile stats (standardised in pipeline)",
             hyperparams=hp, seconds=tm.seconds, notes="Probabilities come from Platt scaling.")


run("svm_rbf", lambda: fit_svm("rbf", "svm_rbf"))
run("svm_linear", lambda: fit_svm("linear", "svm_linear"))

## 6 · Row 9 — Sequence networks (1D-CNN, RNN, LSTM)

| | |
|---|---|
| **Literature anchor** | Kudugunta & Ferrara (2018), a contextual LSTM for bot detection. |
| **Input** | The 168 hourly post counts of each account (`log1p`, standardised). |
| **Output** | Probability of "inauthentic". |
| **Fidelity** | Medium. The cited model reads tweet text plus metadata. Ours reads only the posting timeline, so the architecture family matches but the input does not. |

**Design notes**
- The **CNN** ends in global average pooling, so its last convolutional layer (`features`) keeps a time axis. That is what Grad-CAM needs in step 2, and it is why the CNN is built this way.
- A plain **RNN** can struggle to learn over 168 steps (vanishing gradients). If it ends up near chance, that is a finding, not a bug, and the LSTM is the fair comparison.
- Training: Adam, binary cross-entropy, early stopping on validation loss.

In [ ]:
class SeqCNN(nn.Module):
    """1D CNN over the posting timeline. `features` is the last conv block (the Grad-CAM target layer)."""

    def __init__(self, channels=(16, 32), k=5, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, channels[0], k, padding=k // 2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(channels[0], channels[1], k, padding=k // 2), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling keeps the CAM structure
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(channels[1], 1))

    def forward(self, x):  # x: (batch, 1, 168)
        return self.head(self.pool(self.features(x)).squeeze(-1)).squeeze(-1)


class SeqRNN(nn.Module):
    """Vanilla RNN or LSTM; the last time step's output goes to a linear head."""

    def __init__(self, kind="lstm", hidden=32):
        super().__init__()
        cls = {"rnn": nn.RNN, "lstm": nn.LSTM}[kind]
        self.rnn = cls(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):  # x: (batch, 168, 1)
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(-1)


def fit_seq(kind, name):
    model = SeqCNN() if kind == "cnn" else SeqRNN(kind)
    T = T_seq_cnn if kind == "cnn" else T_seq_rnn
    with Timer() as tm:
        info = train_torch(model, [T[tr_t]], y[tr_idx], [T[va_t]], y[va_idx], epochs=EPOCHS, lr=1e-3)
    score = sigmoid(predict_logits(model, [T]))
    register(name, row=9, family="Deep learning", score=score, score_type="probability", model=model,
             input_view="168 hourly post counts (log1p, standardised)",
             hyperparams=dict(arch=kind, lr=1e-3, batch_size=128, **info), seconds=tm.seconds)


run("cnn1d", lambda: fit_seq("cnn", "cnn1d"))
run("rnn", lambda: fit_seq("rnn", "rnn"))
run("lstm", lambda: fit_seq("lstm", "lstm"))

## 7 · Row 10 — Small transformer on post text

| | |
|---|---|
| **Literature anchor** | Fabien et al. (2020), *BertAA*, which fine-tunes a pretrained BERT for authorship. |
| **Input** | Individual posts, tokenised on whitespace (vocabulary built from training posts only). |
| **Output** | A probability per **post**; an account's score is the **mean** of its six posts' probabilities. |
| **Fidelity** | **Low.** This is a small transformer (2 layers, 64 dimensions) trained **from scratch**, not a fine-tuned pretrained model. Pretrained weights usually need a download from the Hugging Face hub, which may be blocked in restricted environments. Our synthetic posts are random filler words plus a few marker words, so pretrained language knowledge would add little anyway. In the report, call this a simplified stand-in. |

**What to expect.** Marker words (`giveaway`, `idiot`, ...) are a real signal. The decoy hashtag `#trending` is strongly correlated with the label in train and val but not in test, so a model that leans on it shows a **val-to-test AUC drop**. The summary table reports that gap on purpose.

In [ ]:
PAD, CLS, UNK = 0, 1, 2


class PostTransformer(nn.Module):
    """Small transformer encoder. A [CLS] token is prepended and its output is classified."""

    def __init__(self, vocab_size, max_len, d_model=64, nhead=4, n_layers=2, d_ff=128, dropout=0.1):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 1)

    def embed(self, ids):  # (batch, L) -> (batch, d_model): the [CLS] representation
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        h = self.encoder(self.tok(ids) + self.pos(pos), src_key_padding_mask=(ids == PAD))
        return h[:, 0]

    def forward(self, ids):
        return self.head(self.embed(ids)).squeeze(-1)


def build_vocab(token_lists):
    vocab = {"<pad>": PAD, "<cls>": CLS, "<unk>": UNK}
    for toks in token_lists:
        for t in toks:
            vocab.setdefault(t, len(vocab))
    return vocab


# Vocabulary comes from TRAIN accounts' posts only; unseen words in val/test map to <unk>.
VOCAB = build_vocab([t for t, a in zip(post_tokens, post_account) if is_tr[a]])
MAX_LEN = 1 + max(len(t) for t in post_tokens)


def encode(tokens):
    ids = [CLS] + [VOCAB.get(t, UNK) for t in tokens]
    return ids + [PAD] * (MAX_LEN - len(ids))


POST_IDS = torch.as_tensor([encode(t) for t in post_tokens], dtype=torch.long)  # (n_posts, MAX_LEN)
POST_Y = y[post_account]  # each post inherits its account's label
POST_TR, POST_VA = is_tr[post_account], is_va[post_account]
TEXT_STATE = {}  # filled by fit_transformer; reused by the multimodal model


def fit_transformer():
    model = PostTransformer(len(VOCAB), MAX_LEN)
    with Timer() as tm:
        ptr, pva = torch.as_tensor(np.flatnonzero(POST_TR)), torch.as_tensor(np.flatnonzero(POST_VA))
        info = train_torch(model, [POST_IDS[ptr]], POST_Y[POST_TR], [POST_IDS[pva]], POST_Y[POST_VA],
                           epochs=EPOCHS, lr=1e-3, batch_size=256)
    post_prob = sigmoid(predict_logits(model, [POST_IDS]))
    # account score = mean of its posts' probabilities
    score = pd.Series(post_prob).groupby(post_account).mean().reindex(np.arange(n)).to_numpy()
    TEXT_STATE["model"] = model
    register("transformer", row=10, family="Deep learning", score=score, score_type="probability", model=model,
             input_view="post text (whitespace tokens, vocabulary from train)",
             hyperparams=dict(d_model=64, nhead=4, n_layers=2, d_ff=128, lr=1e-3, batch_size=256, **info),
             seconds=tm.seconds, extra={"vocab": VOCAB, "max_len": MAX_LEN},
             notes="Trained from scratch (no pretrained weights). Account score = mean of 6 post probabilities.")


run("transformer", fit_transformer)

## 8 · Row 12 — Graph neural networks (GCN, GAT, GraphSAGE, R-GCN)

| | |
|---|---|
| **Literature anchor** | Feng et al. (2021), *BotRGCN*, a relational GCN for bot detection; TwiBot-22 benchmarks the family. |
| **Input** | Each account's 12 standardised profile stats as node features, plus the interaction graph. |
| **Output** | Probability of "inauthentic" per account (node classification). |
| **Training** | **Transductive** and full-batch: the whole graph and all node features are visible, but the loss uses **train labels only**. Early stopping on validation loss. |
| **Fidelity** | High for GCN, GAT and GraphSAGE (all standard layers). **R-GCN is a weaker match:** BotRGCN uses several relation types (follow, friend, ...), but our graph has a single honest relation, because the edge types are answer-key information and are deliberately hidden. R-GCN here therefore runs with **one relation**. It still has R-GCN's separate self-connection weight, but not its multi-relation structure. |

PyTorch Geometric is imported inside the function, so if it is not installed only these four models fail.

In [ ]:
def make_gnn(kind, d_in, hidden=64, dropout=0.3):
    """Two-layer node classifier. kind in {gcn, gat, sage, rgcn}. Classes are built here so the
    PyTorch Geometric import stays local to this section."""
    from torch_geometric.nn import GATConv, GCNConv, RGCNConv, SAGEConv

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.kind, self.dropout = kind, dropout
            if kind == "gcn":
                self.c1, self.c2 = GCNConv(d_in, hidden), GCNConv(hidden, hidden)
            elif kind == "gat":
                self.c1 = GATConv(d_in, hidden // 4, heads=4)
                self.c2 = GATConv(hidden, hidden // 4, heads=4)
            elif kind == "sage":
                self.c1, self.c2 = SAGEConv(d_in, hidden), SAGEConv(hidden, hidden)
            elif kind == "rgcn":  # one relation type only (see markdown above)
                self.c1 = RGCNConv(d_in, hidden, num_relations=1)
                self.c2 = RGCNConv(hidden, hidden, num_relations=1)
            else:
                raise ValueError(kind)
            self.out = nn.Linear(hidden, 1)

        def conv(self, layer, h, edge_index):
            if self.kind == "rgcn":
                return layer(h, edge_index, torch.zeros(edge_index.size(1), dtype=torch.long, device=h.device))
            return layer(h, edge_index)

        def forward(self, x, edge_index):
            h = F.dropout(x, self.dropout, self.training)
            h = F.relu(self.conv(self.c1, h, edge_index))
            h = F.dropout(h, self.dropout, self.training)
            h = F.relu(self.conv(self.c2, h, edge_index))
            return self.out(h).squeeze(-1)

    return Net()


def fit_gnn(kind, name):
    model = make_gnn(kind, T_tab.shape[1])
    x, ei = T_tab.to(DEVICE), EDGE_INDEX.to(DEVICE)
    with Timer() as tm:
        info, logits = train_fullbatch(model, lambda m: m(x, ei), y, is_tr, is_va, epochs=GNN_EPOCHS)
    register(name, row=12, family="Deep learning", score=sigmoid(logits), score_type="probability", model=model,
             input_view="12 standardised profile stats as node features + undirected interaction graph",
             hyperparams=dict(kind=kind, hidden=64, dropout=0.3, lr=0.01, **info), seconds=tm.seconds,
             notes="Transductive; train labels only. R-GCN uses a single relation type." if kind == "rgcn"
             else "Transductive; train labels only.")


run("gcn", lambda: fit_gnn("gcn", "gcn"))
run("gat", lambda: fit_gnn("gat", "gat"))
run("graphsage", lambda: fit_gnn("sage", "graphsage"))
run("rgcn", lambda: fit_gnn("rgcn", "rgcn"))

## 9 · Row 13 — Autoencoders (unsupervised anomaly detection)

| | |
|---|---|
| **Literature anchor** | MulBot (Mannocci et al. 2022) and RTbust (Mazza et al. 2019), which use LSTM autoencoders on time series and flag accounts that reconstruct badly. |
| **Training** | **No labels.** The autoencoder is trained to reconstruct the **training accounts as they come** (about 30% are inauthentic), with early stopping on validation reconstruction error. |
| **Output** | An **anomaly score** = per-account reconstruction error (higher = more anomalous). It is not a probability, so log loss and Brier score are left empty. |
| **Threshold** | Chosen on validation labels with the same F1 rule as every model. Training uses no labels, but picking the cut-off does. |
| **Two versions** | A dense autoencoder on the 12 profile stats, and an LSTM autoencoder on the posting timeline (the version closest to the cited papers). |
| **Caveat** | An autoencoder trained on a 30%-contaminated population learns to reconstruct *both* kinds of account, so weak performance is expected. |

In [ ]:
class DenseAE(nn.Module):
    def __init__(self, d_in, bottleneck=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in, 8), nn.ReLU(), nn.Linear(8, bottleneck))
        self.dec = nn.Sequential(nn.Linear(bottleneck, 8), nn.ReLU(), nn.Linear(8, d_in))

    def forward(self, x):
        return self.dec(self.enc(x))


class LSTMAutoencoder(nn.Module):
    """Encode the whole timeline into the final hidden state, then decode it back step by step."""

    def __init__(self, hidden=32):
        super().__init__()
        self.enc = nn.LSTM(1, hidden, batch_first=True)
        self.dec = nn.LSTM(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):  # x: (batch, 168, 1)
        _, (h, _) = self.enc(x)
        z = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)  # (batch, 168, hidden)
        d, _ = self.dec(z)
        return self.out(d)


def fit_autoencoder(kind, name):
    if kind == "dense":
        model, T, view = DenseAE(T_tab.shape[1]), T_tab, "12 standardised profile stats"
    else:
        model, T, view = LSTMAutoencoder(), T_seq_rnn, "168 hourly post counts (log1p, standardised)"
    with Timer() as tm:
        info = train_autoencoder(model, T[tr_t], T[va_t], epochs=EPOCHS, lr=1e-3)
    score = recon_error(model, T)
    register(name, row=13, family="Deep learning", score=score, score_type="anomaly", model=model,
             input_view=view, hyperparams=dict(kind=kind, lr=1e-3, **info), seconds=tm.seconds,
             supervised=False, notes="Trained without labels on all train accounts; score = reconstruction MSE.")


run("autoencoder_dense", lambda: fit_autoencoder("dense", "autoencoder_dense"))
run("autoencoder_lstm", lambda: fit_autoencoder("lstm", "autoencoder_lstm"))

## 10 · Row 14 — Multimodal fusion (text + profile stats + graph)

| | |
|---|---|
| **Literature anchor** | Lei et al. (2023), *BIC*, a text-graph interaction model for bot detection. |
| **Input** | Three modalities per account: (1) a **text embedding**, the mean [CLS] vector of the account's posts from the transformer in section 7; (2) the **12 profile stats**; (3) the **interaction graph**. |
| **Architecture** | Late fusion: the text and profile vectors are projected and concatenated, two GCN layers then pass messages over the graph, and the classifier reads the node's own fused vector together with its graph-context vector. |
| **Output** | Probability of "inauthentic". |
| **Fidelity** | **Low.** BIC has a dedicated text-graph interaction module and a semantic-consistency component. This is a simple fusion network in the same spirit. It also inherits the section-7 transformer **frozen**: if that model failed, this one is skipped. In the report, call it a simplified stand-in. |

In [ ]:
def make_fusion(d_tab, d_txt, hidden=64, dropout=0.3):
    from torch_geometric.nn import GCNConv

    class Fusion(nn.Module):
        def __init__(self):
            super().__init__()
            self.dropout = dropout
            self.tab_proj = nn.Linear(d_tab, hidden // 2)
            self.txt_proj = nn.Linear(d_txt, hidden // 2)
            self.c1, self.c2 = GCNConv(hidden, hidden), GCNConv(hidden, hidden)
            self.out = nn.Linear(2 * hidden, 1)

        def forward(self, x_tab, x_txt, edge_index):
            h0 = F.relu(torch.cat([self.tab_proj(x_tab), self.txt_proj(x_txt)], dim=1))  # fused node vector
            h = F.dropout(h0, self.dropout, self.training)
            h = F.relu(self.c1(h, edge_index))
            h = F.dropout(h, self.dropout, self.training)
            h = F.relu(self.c2(h, edge_index))
            return self.out(torch.cat([h0, h], dim=1)).squeeze(-1)  # own features + graph context

    return Fusion()


@torch.no_grad()
def account_text_embeddings(text_model):
    """Mean [CLS] embedding of each account's posts, from the (frozen) section-7 transformer."""
    text_model.eval().to(DEVICE)
    chunks = [text_model.embed(POST_IDS[i:i + 2048].to(DEVICE)).cpu() for i in range(0, len(POST_IDS), 2048)]
    post_emb = torch.cat(chunks).numpy()
    emb = np.zeros((n, post_emb.shape[1]))
    np.add.at(emb, post_account, post_emb)
    emb /= np.bincount(post_account, minlength=n)[:, None]
    return emb


def fit_multimodal():
    if "model" not in TEXT_STATE:
        raise RuntimeError("needs the transformer from section 7, which did not run successfully")
    emb = account_text_embeddings(TEXT_STATE["model"])
    sc = StandardScaler().fit(emb[is_tr])
    x_txt = torch.as_tensor(sc.transform(emb), dtype=torch.float32).to(DEVICE)
    x_tab, ei = T_tab.to(DEVICE), EDGE_INDEX.to(DEVICE)
    model = make_fusion(x_tab.shape[1], x_txt.shape[1])
    with Timer() as tm:
        info, logits = train_fullbatch(model, lambda m: m(x_tab, x_txt, ei), y, is_tr, is_va, epochs=GNN_EPOCHS)
    register("multimodal_fusion", row=14, family="Deep learning", score=sigmoid(logits),
             score_type="probability", model=model,
             input_view="text embedding (frozen transformer) + 12 profile stats + interaction graph",
             hyperparams=dict(hidden=64, dropout=0.3, lr=0.01, **info), seconds=tm.seconds,
             extra={"text_scaler": sc}, notes="Text encoder is frozen from section 7. Simplified stand-in for BIC.")


run("multimodal_fusion", fit_multimodal)

## 11 · Row 19 — Matrix decomposition + logistic regression (PCA, NMF, truncated SVD)

| | |
|---|---|
| **Literature anchor** | Abbasi & Chen, *Writeprints*, whose visualisation is PCA on stylometric features. |
| **Why a pipeline** | PCA and its relatives only reduce dimensions; they do not predict anything. Each is placed in front of a logistic regression, so what is explained in step 2 is *the components plus the classifier*. |
| **Input** | The 12 profile stats (the label depends on them). NMF needs non-negative input, so that pipeline scales to [0, 1] first. |
| **Output** | Probability of "inauthentic". |
| **Fidelity** | Medium: the decompositions are standard, but the literature uses PCA for *visualisation*, so adding a classifier is our adaptation. |
| **Note** | These are the most interpretable models in this list. The number of components (6) is fixed rather than tuned, and the explained variance is recorded in the notes. |

In [ ]:
def fit_decomp(kind, name):
    n_comp = 6
    if kind == "pca":
        reducer, prep = PCA(n_components=n_comp, random_state=SEED), StandardScaler()
    elif kind == "svd":
        reducer, prep = TruncatedSVD(n_components=n_comp, random_state=SEED), StandardScaler()
    else:  # nmf needs non-negative input
        reducer, prep = NMF(n_components=n_comp, init="nndsvda", max_iter=1000, random_state=SEED), MinMaxScaler(clip=True)
    with Timer() as tm:
        m = Pipeline([("prep", prep), ("reduce", reducer), ("clf", LogisticRegression(max_iter=2000))])
        m.fit(X_df.loc[is_tr], y[is_tr])
    note = ""
    fitted = m.named_steps["reduce"]
    if hasattr(fitted, "explained_variance_ratio_"):
        note = f"explained variance of {n_comp} components: {fitted.explained_variance_ratio_.sum():.2f}"
    register(name, row=19, family="Unsupervised + classifier", score=m.predict_proba(X_df)[:, 1],
             score_type="probability", model=m, input_view="12 profile stats", supervised=True,
             hyperparams=dict(reducer=kind, n_components=n_comp), seconds=tm.seconds, notes=note)


run("pca_logreg", lambda: fit_decomp("pca", "pca_logreg"))
run("nmf_logreg", lambda: fit_decomp("nmf", "nmf_logreg"))
run("svd_logreg", lambda: fit_decomp("svd", "svd_logreg"))

## 12 · Row 20 — Outlier detectors (Isolation Forest, LOF, one-class SVM)

| | |
|---|---|
| **Literature anchor** | The coordinated fake-follower benchmark (*EPJ Data Science*), which compares Isolation Forest and LOF. |
| **Training** | **No labels**, fitted on the standardised profile stats of the training accounts. |
| **Output** | An **anomaly score** (higher = more anomalous), not a probability. |
| **Threshold** | Chosen on validation labels, as for every model. |
| **Details** | LOF is fitted in *novelty* mode so it can score accounts it has not seen. The one-class SVM's `nu` is set to 0.3, which approximates the share of inauthentic accounts; that is a mild use of what we know about the data, noted in the registry. |
| **Expectation** | Weak scores. Fake accounts here are 30% of the population, not rare outliers, and the cited paper also reports AUCs near 0.5 to 0.6. A near-chance detector is not worth explaining; the summary flags it. |

In [ ]:
def fit_outlier(kind, name):
    Xtr = X_tab_std[is_tr]
    if kind == "iforest":
        m, hp = IsolationForest(n_estimators=N_EST, random_state=SEED, n_jobs=-1), dict(n_estimators=N_EST)
    elif kind == "lof":
        hp = dict(n_neighbors=35, novelty=True)
        m = LocalOutlierFactor(**hp)
    else:
        hp = dict(kernel="rbf", nu=0.3, gamma="scale")
        m = OneClassSVM(**hp)
    with Timer() as tm:
        m.fit(Xtr)  # no labels
    score = -m.score_samples(X_tab_std)  # sklearn: higher score_samples = more normal, so negate
    register(name, row=20, family="Unsupervised / semi-supervised", score=score, score_type="anomaly",
             model=Pipeline([("scale", scaler_tab), ("detector", m)]),
             input_view="12 profile stats (standardised with train statistics)", hyperparams=hp,
             seconds=tm.seconds, supervised=False,
             notes="Fitted without labels. nu=0.3 approximates the inauthentic share." if kind == "ocsvm" else "Fitted without labels.")


run("isolation_forest", lambda: fit_outlier("iforest", "isolation_forest"))
run("lof", lambda: fit_outlier("lof", "lof"))
run("one_class_svm", lambda: fit_outlier("ocsvm", "one_class_svm"))

## 13 · Row 21 — Embeddings + logistic regression (word2vec, doc2vec, node2vec, DeepWalk)

| | |
|---|---|
| **Literature anchor** | word2vec (Mikolov et al. 2013), node2vec (Grover & Leskovec 2016). DeepWalk is node2vec with its walk bias switched off (`p = q = 1`). |
| **Two-stage models** | An embedding is learned **without labels** from the **training accounts only** (text for word2vec and doc2vec, the graph for node2vec and DeepWalk). A logistic regression on those embeddings is then trained on the training labels. The embedding alone is not a predictor. |
| **Output** | Probability of "inauthentic" from the logistic regression. |
| **Fidelity** | High for the embedding methods. A single embedding dimension has no meaning, so step 2 will explain what the embedding captures and what the classifier uses. |
| **Details** | word2vec: an account's vector is the mean of its words' vectors. doc2vec: training accounts are learned directly, and val/test accounts are *inferred* with the trained model. node2vec / DeepWalk: the graph structure (no labels) of all accounts is used, which is standard for unsupervised graph embedding. |
| **Dependencies** | `gensim` for the text embeddings and the `node2vec` package for graph walks. If the `node2vec` package clashes with your gensim or NumPy version, that cell fails alone and is recorded. |

In [ ]:
def lr_on_embeddings(E):
    clf = Pipeline([("scale", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))])
    clf.fit(E[is_tr], y[is_tr])
    return clf, clf.predict_proba(E)[:, 1]


def fit_word2vec():
    from gensim.models import Word2Vec

    dim = 50
    train_sents = [t for t, a in zip(post_tokens, post_account) if is_tr[a]]  # unsupervised, train accounts only
    with Timer() as tm:
        w2v = Word2Vec(sentences=train_sents, vector_size=dim, window=5, min_count=1, workers=1,
                       epochs=EMB_EPOCHS, seed=SEED)
        E = np.vstack([np.mean([w2v.wv[t] for t in toks if t in w2v.wv], axis=0) if toks else np.zeros(dim)
                       for toks in account_tokens])
        clf, score = lr_on_embeddings(E)
    register("word2vec_logreg", row=21, family="Unsupervised + classifier", score=score, score_type="probability",
             model=w2v, extra={"classifier": clf}, input_view="post text -> mean word2vec vector per account",
             hyperparams=dict(vector_size=dim, window=5, epochs=EMB_EPOCHS), seconds=tm.seconds)


def fit_doc2vec():
    from gensim.models.doc2vec import Doc2Vec, TaggedDocument

    dim = 50
    docs = [TaggedDocument(account_tokens[i], [int(i)]) for i in tr_idx]  # train accounts only
    with Timer() as tm:
        d2v = Doc2Vec(docs, vector_size=dim, window=5, min_count=1, workers=1, epochs=EMB_EPOCHS, seed=SEED)
        E = np.vstack([d2v.dv[int(i)] if is_tr[i] else d2v.infer_vector(account_tokens[i]) for i in range(n)])
        clf, score = lr_on_embeddings(E)
    register("doc2vec_logreg", row=21, family="Unsupervised + classifier", score=score, score_type="probability",
             model=d2v, extra={"classifier": clf},
             input_view="post text -> doc2vec vector per account (val/test inferred)",
             hyperparams=dict(vector_size=dim, window=5, epochs=EMB_EPOCHS), seconds=tm.seconds)


def fit_graph_embedding(p, q, name):
    import networkx as nx
    from node2vec import Node2Vec

    dim = 32
    G = nx.Graph()
    G.add_nodes_from(range(n))  # keep isolated accounts as nodes
    G.add_edges_from(zip(src.tolist(), dst.tolist()))
    with Timer() as tm:
        walker = Node2Vec(G, dimensions=dim, walk_length=20, num_walks=N2V_WALKS, p=p, q=q, workers=1, seed=SEED)
        w2v = walker.fit(window=5, min_count=1, epochs=EMB_EPOCHS)
        E = np.vstack([w2v.wv[str(i)] if str(i) in w2v.wv else np.zeros(dim) for i in range(n)])
        clf, score = lr_on_embeddings(E)
    register(name, row=21, family="Unsupervised + classifier", score=score, score_type="probability",
             model=w2v, extra={"classifier": clf}, input_view="interaction graph -> random-walk embedding per account",
             hyperparams=dict(dimensions=dim, walk_length=20, num_walks=N2V_WALKS, p=p, q=q),
             seconds=tm.seconds, notes="DeepWalk = node2vec with p = q = 1." if name == "deepwalk_logreg" else "")


run("word2vec_logreg", fit_word2vec)
run("doc2vec_logreg", fit_doc2vec)
run("node2vec_logreg", lambda: fit_graph_embedding(p=0.5, q=2.0, name="node2vec_logreg"))
run("deepwalk_logreg", lambda: fit_graph_embedding(p=1.0, q=1.0, name="deepwalk_logreg"))

## 14 · Row 22 — Label propagation (label spreading, belief propagation)

| | |
|---|---|
| **Literature anchor** | Gong, Frank & Mittal (2014), *SybilBelief*: a Markov random field with loopy belief propagation, seeded by a few known benign or fake accounts. |
| **Setup** | Only a **small set of labels** is revealed: `LP_LABELED_FRAC` of the *training* accounts (default 20%). Everyone else, including all val and test accounts, starts unlabeled, and the labels spread along the graph. |
| **Input** | The interaction graph and the revealed labels. **No profile stats, text or timelines.** |
| **Output** | A score between 0 and 1 per account (belief that it is inauthentic). It is graph-based and not calibrated, so it is treated as a score, not a probability. |
| **Two versions** | (a) **Label spreading** (Zhou et al. 2004): a standard iterative scheme. (b) **Belief propagation**: our own loopy BP on a pairwise MRF, which is the faithful SybilBelief-style method. No standard package implements it, so it is written out here. |
| **Expectation** | The graph only carries signal for trolls (their rings are dense); bots are linked to random authentic accounts. Both methods should be clearly better than chance, but far from the feature-based models. |
| **Not fitted in the usual sense** | There are no learned weights. The "model" stored is the setup (seed accounts and parameters) plus the outputs. |

In [ ]:
def build_adjacency():
    A = sparse.coo_matrix((np.ones(2 * len(src)), (np.r_[src, dst], np.r_[dst, src])), shape=(n, n)).tocsr()
    A.data[:] = 1.0  # collapse any duplicate edges
    return A


def pick_seeds():
    rng = np.random.default_rng(SEED)
    k = max(1, int(LP_LABELED_FRAC * is_tr.sum()))
    return rng.choice(tr_idx, size=k, replace=False)


def fit_label_spreading():
    A, seeds = build_adjacency(), pick_seeds()
    alpha, n_iter = 0.8, 100
    with Timer() as tm:
        deg = np.asarray(A.sum(1)).ravel()
        dinv = sparse.diags(1.0 / np.sqrt(np.maximum(deg, 1.0)))
        S = dinv @ A @ dinv  # symmetric normalisation
        Y0 = np.zeros((n, 2))
        Y0[seeds, y[seeds]] = 1.0
        Fm = Y0.copy()
        for _ in range(n_iter):
            Fm = alpha * (S @ Fm) + (1 - alpha) * Y0
        prior = float(y[seeds].mean())
        tot = Fm.sum(1)
        score = np.where(tot > 1e-12, Fm[:, 1] / np.maximum(tot, 1e-12), prior)  # unreachable nodes -> prior
    register("label_spreading", row=22, family="Unsupervised / semi-supervised", score=score, score_type="score",
             model={"seed_accounts": seeds, "alpha": alpha, "n_iter": n_iter}, input_view="interaction graph + a few known labels",
             hyperparams=dict(alpha=alpha, n_iter=n_iter, labeled_frac=LP_LABELED_FRAC, n_seeds=int(len(seeds))),
             seconds=tm.seconds, supervised=True,
             notes="Graph-only. Score is a normalised propagated label mass, not a calibrated probability.")


def fit_belief_propagation():
    seeds = pick_seeds()
    homophily, damping, n_iter, seed_strength = 0.1, 0.5, 30, 0.9
    prior = float(y[seeds].mean())
    with Timer() as tm:
        # --- node potentials: seeds are confident, everyone else gets the class prior
        phi = np.tile([1 - prior, prior], (n, 1))
        phi[seeds] = np.where(y[seeds, None] == 1, [1 - seed_strength, seed_strength], [seed_strength, 1 - seed_strength])
        log_phi = np.log(phi)
        # --- edge potential: neighbours tend to share a class (weak homophily keeps BP from over-smoothing)
        psi = np.array([[0.5 + homophily, 0.5 - homophily], [0.5 - homophily, 0.5 + homophily]])
        # --- directed edges: e = (a -> b); the reverse of edge e is e + E/2 (mod E)
        a_idx, b_idx = np.r_[src, dst], np.r_[dst, src]
        E = len(a_idx)
        rev = (np.arange(E) + E // 2) % E
        log_m = np.zeros((E, 2))  # log-message a -> b, over the class of b
        for _ in range(n_iter):
            incoming = np.zeros((n, 2))
            np.add.at(incoming, b_idx, log_m)  # sum of log-messages arriving at each node
            cav = log_phi[a_idx] + incoming[a_idx] - log_m[rev]  # belief at a, excluding b's own message
            p = np.exp(cav - cav.max(1, keepdims=True))
            p /= p.sum(1, keepdims=True)
            new = np.log(p @ psi)  # sum over a's class -> message over b's class
            new -= np.log(np.exp(new).sum(1, keepdims=True))
            log_m = damping * log_m + (1 - damping) * new
        incoming = np.zeros((n, 2))
        np.add.at(incoming, b_idx, log_m)
        logb = log_phi + incoming
        logb -= logb.max(1, keepdims=True)
        belief = np.exp(logb)
        score = belief[:, 1] / belief.sum(1)
    register("belief_propagation", row=22, family="Unsupervised / semi-supervised", score=score, score_type="score",
             model={"seed_accounts": seeds, "homophily": homophily, "damping": damping, "n_iter": n_iter},
             input_view="interaction graph + a few known labels",
             hyperparams=dict(homophily=homophily, damping=damping, n_iter=n_iter, seed_strength=seed_strength,
                              labeled_frac=LP_LABELED_FRAC, n_seeds=int(len(seeds))),
             seconds=tm.seconds, supervised=True,
             notes="SybilBelief-style loopy BP on a pairwise MRF; written from scratch. Belief is not calibrated.")


run("label_spreading", fit_label_spreading)
run("belief_propagation", fit_belief_propagation)

## 15 · Summary: all models side by side

One table with every model's **test** accuracy measures, the **val** and **train** ROC-AUC for diagnosing overfitting, and a status flag.

How to read it:

- **`status`** (from test ROC-AUC): `near chance` (< 0.55), `weak` (0.55 to 0.65), `ok` (≥ 0.65). A model flagged `near chance` learned little, so explaining it is not informative. Keep it in the report, but read its step-2 results with that in mind.
- **`val − test AUC`**: a large positive gap means the model did well where the decoy hashtag still worked (train and val) and worse where it did not (test). That is expected for text models that rely on `#trending`.
- **`train − test AUC`**: a large gap means overfitting.
- **Reference**: the best possible AUC from the profile stats alone is printed above the table. It applies only to models that read those stats.

In [ ]:
def build_metrics_table():
    rows = []
    for name, r in RESULTS.items():
        te, va, tr = r["metrics"]["test"], r["metrics"]["val"], r["metrics"]["train"]
        auc = te["roc_auc"]
        rows.append({
            "model": name, "sheet_row": r["row"], "family": r["family"], "unit": r["unit"],
            "supervised": r["supervised"], "score_type": r["score_type"],
            "test_roc_auc": auc, "auc_ci_low": r["test_auc_ci"][0], "auc_ci_high": r["test_auc_ci"][1],
            "test_pr_auc": te["pr_auc"], "test_f1": te["f1"], "test_precision": te["precision"],
            "test_recall": te["recall"], "test_balanced_acc": te["balanced_accuracy"],
            "test_accuracy": te["accuracy"], "test_log_loss": te["log_loss"], "test_brier": te["brier"],
            "val_roc_auc": va["roc_auc"], "train_roc_auc": tr["roc_auc"],
            "val_minus_test_auc": va["roc_auc"] - auc, "train_minus_test_auc": tr["roc_auc"] - auc,
            "threshold": r["threshold"], "train_seconds": r["train_seconds"],
            "status": "baseline" if r["row"] == 0 else ("near chance" if auc < CHANCE_BAND else "weak" if auc < WEAK_BAND else "ok"),
            "input_view": r["input_view"],
        })
    return pd.DataFrame(rows)


metrics_df = build_metrics_table()
print(f"Reference ceiling (profile stats only): test ROC-AUC = {CEILING_TEST_AUC:.3f}")
print(f"Fitted: {len(RESULTS) - 1} models + baseline | Failed: {len(FAILURES)} {sorted(FAILURES) if FAILURES else ''}")

show = ["model", "sheet_row", "test_roc_auc", "auc_ci_low", "auc_ci_high", "test_pr_auc", "test_f1",
        "test_accuracy", "val_minus_test_auc", "train_minus_test_auc", "status"]
account_level = metrics_df.sort_values("test_roc_auc", ascending=False)
display(account_level[show].round(3).reset_index(drop=True))

In [ ]:
# Test ROC-AUC with bootstrap 95% intervals, account-level models only.
plot_df = account_level[account_level["sheet_row"] > 0].iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 0.32 * len(plot_df) + 1.5))
ax.barh(plot_df["model"], plot_df["test_roc_auc"] - 0.5, left=0.5, color="#4C78A8", height=0.65)
ax.errorbar(plot_df["test_roc_auc"], plot_df["model"],
            xerr=[plot_df["test_roc_auc"] - plot_df["auc_ci_low"], plot_df["auc_ci_high"] - plot_df["test_roc_auc"]],
            fmt="none", ecolor="black", capsize=2, lw=1)
ax.axvline(0.5, color="gray", lw=1)
ax.axvline(CEILING_TEST_AUC, color="#E45756", lw=1, ls="--", label=f"profile-stats ceiling ({CEILING_TEST_AUC:.2f})")
ax.set_xlim(0.4, 1.0)
ax.set_xlabel("Test ROC-AUC (0.5 = chance)")
ax.set_title("Account-level models on held-out test accounts")
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

### Save everything for step 2

In [ ]:
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)
pd.concat(PREDS.values()).to_csv(OUT_DIR / "predictions_all_accounts.csv", index=False)

registry = {name: {k: v for k, v in r.items()} for name, r in RESULTS.items()}
(OUT_DIR / "registry.json").write_text(json.dumps(registry, indent=2, default=str))
(OUT_DIR / "failures.json").write_text(json.dumps(FAILURES, indent=2))
(OUT_DIR / "environment.json").write_text(json.dumps(ENV, indent=2))

print("saved to", OUT_DIR)
for name, f in FAILURES.items():
    print(f"\nFAILED: {name}\n{f['error']}")

## Known limitations to keep in mind when reading the results

1. **Synthetic data.** Results say how well each method works on data with a *known, simple* structure. They are not estimates of real-world performance.
2. **Departures from the literature** (see each section): the transformer (no pretrained weights), the multimodal model (simple fusion, frozen text encoder), R-GCN (one relation type), the sequence networks (posting timeline, not tweet text), and the PCA / NMF / SVD pipelines (a classifier was added).
3. **Unsupervised models and labels.** Threshold selection uses validation labels for the anomaly-score models, and the one-class SVM's `nu` approximates the known inauthentic share.
4. **Single run.** One seed and one train/val/test split. Small AUC differences between models are within the bootstrap intervals and should not be over-read.
5. **Untested hyperparameters.** Values are reasonable defaults, not tuned. Tuning is not the goal here.

## Next: step 2

Load the stored models from `outputs/step1_fit/models/` and run the explainability methods (feature importance, PDP / ALE, LIME, SHAP, counterfactuals, Grad-CAM and variants, GNNExplainer), scoring each explanation against the answer key in `truth/`.